# Notebook 01 — Data Exploration (EDA)

Objective: load the raw dataset, inspect it, document the class imbalance, and investigate duplicates and missing values.

**No model training. No preprocessing. No data leakage.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config import RAW_CSV, TARGET_COL, FIGURES_DIR
from src.data_processing import (
    load_raw_data,
    dataset_overview,
    investigate_duplicates,
    check_missing_values,
    class_distribution,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

## 1. Load Raw Dataset

In [ ]:
df = load_raw_data(RAW_CSV)
df.head()

## 2. Dataset Overview

In [ ]:
overview = dataset_overview(df)
df.describe()

## 3. Missing Values

In [ ]:
missing = check_missing_values(df)

## 4. Duplicate Investigation

Do NOT automatically remove — investigate first.

In [ ]:
dup_df = investigate_duplicates(df)
dup_df.head(10)

## 5. Class Distribution (Imbalance)

In [ ]:
dist = class_distribution(df)

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['steelblue', 'tomato']
dist['count'].plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_title('Transaction Class Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_xticklabels(dist.index, rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150)
plt.show()
print('Class imbalance: fraud =', round(dist.loc['Fraudulent (1)','percent'], 4), '%')

## 6. Feature Distributions — Amount and Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, grp in df.groupby(TARGET_COL):
    name = 'Fraudulent' if label == 1 else 'Legitimate'
    grp['Amount'].hist(ax=axes[0], bins=50, alpha=0.6, label=name, density=True)
    grp['Time'].hist(ax=axes[1], bins=50, alpha=0.6, label=name, density=True)

axes[0].set_title('Amount Distribution by Class')
axes[0].set_xlabel('Amount')
axes[0].legend()
axes[1].set_title('Time Distribution by Class')
axes[1].set_xlabel('Time (seconds)')
axes[1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'amount_time_distributions.png', dpi=150)
plt.show()

## 7. V-Feature Distributions (sample)

V1–V28 are anonymized PCA-transformed features. We show a sample to illustrate distributional differences between classes.

In [ ]:
sample_v_cols = ['V1', 'V2', 'V3', 'V4', 'V10', 'V12', 'V14', 'V17']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

legit = df[df[TARGET_COL] == 0]
fraud = df[df[TARGET_COL] == 1]

for i, col in enumerate(sample_v_cols):
    axes[i].hist(legit[col], bins=50, alpha=0.6, label='Legitimate', density=True, color='steelblue')
    axes[i].hist(fraud[col], bins=50, alpha=0.6, label='Fraudulent', density=True, color='tomato')
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

plt.suptitle('Distribution of Selected V-features by Class', y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'v_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Correlation Heatmap

In [ ]:
corr = df.corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.1, ax=ax, annot=False)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()

## 9. EDA Summary

Document findings here before proceeding to Notebook 02.

- **Missing values**: (fill in from output above)
- **Duplicates**: (fill in count and class breakdown)
- **Class imbalance**: ~0.17% fraud — accuracy alone is insufficient
- **Amount / Time**: (fill in observations)
- **V-features**: anonymized — distributional differences visible between classes for several features (V14, V12, V10 typically most discriminative)